In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from scipy.optimize import fsolve
%run IPR.ipynb

In [3]:
# Input Datas for VLP
Pwh_VLP = 100
T = 150 + 460
d = 2.441
API = 35
GLR = 273
Bw = 1.01
fo = 1.0
fw = 0.0
Yg = 0.75 # gas gravity
Yo = 0.55 # Oil gravity
Yw = 1.05 # water gravity
depth = 5000

In [6]:
def Poettmann_and_Carpenter_Method(Pwh, T, d, API, GLR, Bw, fo, fw, Yg, Yo, Yw, Ql, depth):
    i = 0
    Pwf = np.zeros_like(Ql)

    for i in range(len(Ql)):
         # Calculated Datas
         Tpc = 168 + 325 * Yg - 12.5 * Yg**2
         Tpr = T / Tpc
         Ppc = 677 + 15.0 * Yg - 37.5 * Yg**2
         M = 350.376 * (fo * Yo + fw * Yw) + 0.0763 * Yg * GLR
         Dvp = 176.844 * (10**-6) * (M * Ql[i] / d)
         Ef = 46.0115 * Dvp**-3.092368 + 60.37 * (10**-6) * (Dvp**-1) + 0.00524355
         K_bar = (3.3567 * (10**-6) * (Ql[i]**2) * (M**2) * Ef) / (d**5)


         # Initialize an empty list to store data
         data_list = []

         # Moving till depth
         delta_p = 0
         H = 0
         DTT = 0
         P = Pwh
         while DTT < depth - 1:
              if DTT < depth - 50:
                  P += delta_p
                  P_avg = P - delta_p / 2
                  Rs = Yg * ((P_avg / 18.2 + 1.4) * 10**(0.0125 * API - 0.00091 * (T - 460)))**1.2048
                  Bo = 0.9759 + 0.000120 * (Rs * ((Yg / Yo)**0.5) + 1.25 * (T - 460))**1.2
                  Ppr = P_avg / Ppc
                  Z = 1.008505 + 0.04623 * (Ppr / Tpr) + (0.862707 * Ppr**1.368627) / (10**(0.636778 * Tpr)) - (2.324825 * Ppr) / (10**(0.649787 * Tpr))
                  Bg = 0.02827 * Z * T / P_avg
                  Row_bar = M / (5.614 * (fo * Bo + fw * Bw) + Bg * (GLR - fo * Rs))
                  delta_h = 144.9 * delta_p / (Row_bar + K_bar / Row_bar)
                  delta_p = 5
                  DTT += delta_h
                  # Append a dictionary to the list
                  data_list.append({
                      'Assumed Pressure (psi)': P,
                      'Avg. Pressure (psi)': P_avg,
                      'Rs': Rs,
                      'Bo': Bo,
                      'Ppr': Ppr,
                      'Z': Z,
                      'Bg': Bg,
                      'Avg. Density': Row_bar,
                      'Increment in Depth': delta_h,
                      'Depth to Top': DTT
                                    })

              else:
                  delta_p = 1
                  P += delta_p
                  P_avg = P - delta_p / 2
                  Rs = Yg * ((P_avg / 18.2 + 1.4) * 10**(0.0125 * API - 0.00091 * (T - 460)))**1.2048
                  Bo = 0.9759 + 0.000120 * (Rs * ((Yg / Yo)**0.5) + 1.25 * (T - 460))**1.2
                  Ppr = P_avg / Ppc
                  Z = 1.008505 + 0.04623 * (Ppr / Tpr) + (0.862707 * Ppr**1.368627) / (10**(0.636778 * Tpr)) - (2.324825 * Ppr) / (10**(0.649787 * Tpr))
                  Bg = 0.02827 * Z * T / P_avg
                  Row_bar = M / (5.614 * (fo * Bo + fw * Bw) + Bg * (GLR - fo * Rs))
                  delta_h = 144.9 * delta_p / (Row_bar + K_bar / Row_bar)
                  DTT += delta_h
                  # Append a dictionary to the list
                  data_list.append({
                        'Assumed Pressure (psi)': P,
                        'Avg. Pressure (psi)': P_avg,
                        'Rs': Rs,
                        'Bo': Bo,
                        'Ppr': Ppr,
                        'Z': Z,
                        'Bg': Bg,
                        'Avg. Density': Row_bar,
                        'Increment in Depth': delta_h,
                        'Depth to Top': DTT
                                     })

         df = pd.DataFrame(data_list)
         Pwf[i] = df['Assumed Pressure (psi)'].iloc[-1]
    return Pwf